In [3]:
!pip install lxml


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd 
import numpy as np

In [5]:
link  = 'https://en.wikipedia.org/wiki/List_of_American_films_of_2018'

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
all_tables = pd.read_html(link, header=0, storage_options=headers)
df1 = all_tables[2]
df2 = all_tables[3]
df3 = all_tables[4]
df4 = all_tables[5]

In [6]:
df4

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,O C T O B E R,5,Venom,Columbia Pictures / Marvel Entertainment,"Ruben Fleischer (director); Jeff Pinkner, Scot...",[181]
1,O C T O B E R,5,A Star Is Born,Warner Bros. Pictures / Peters Entertainment,Bradley Cooper (director/screenplay); Eric Rot...,[182]
2,O C T O B E R,5,The Great Buster: A Celebration,Cohen Media Group,Peter Bogdanovich (director/screenplay),[183]
3,O C T O B E R,12,First Man,Universal Pictures / DreamWorks Pictures / Amb...,Damien Chazelle (director); Josh Singer (scree...,[184]
4,O C T O B E R,12,Bad Times at the El Royale,20th Century Fox,Drew Goddard (director/screenplay); Jeff Bridg...,[185]
...,...,...,...,...,...,...
59,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237]
60,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141]
61,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116]
62,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206]


In [7]:
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2]
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3]
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4]
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5]
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6]
...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237]
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141]
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116]
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206]


In [8]:
# Got API key form Moviedb
from tmdbv3api import TMDb
import json
import requests
tmdb = TMDb()
tmdb.api_key = 'ENTER YOUR TMDB API'

In [9]:
# Getting Genres
# from requests import request
from tmdbv3api import Movie
tmdb_movie = Movie()
def get_genre(x):
    genres = []
    result = tmdb_movie.search(x)
    movie_id = result[0].id
    response  = requests.get('https://api.themoviedb.org/3/movie/{}?api_key={}'.format(movie_id,tmdb.api_key))
    data_json = response.json()
    if data_json['genres']:
        genre_str = " "
        for i in range(0, len(data_json['genres'])):
            genres.append(data_json['genres'][i]['name'])
        return genre_str.join(genres)
    else:
        np.nan

In [10]:
df['genres'] = df['Title'].map(lambda x: get_genre(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Drama Mystery
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller
...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Drama Comedy
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History


In [11]:
def get_movie_details(x):
    result = tmdb_movie.search(x)

    if not result:
        return pd.Series([np.nan, np.nan, np.nan])
        
    # 2. Get the movie ID and fetch the full details
    movie_id = result[0].id
    response = requests.get(f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={tmdb.api_key}')
    data_json = response.json()
    
    language = data_json.get('original_language', np.nan)
    duration = data_json.get('runtime', np.nan)
    tmdb_score = data_json.get('vote_average', np.nan)
    
    return pd.Series([language, duration, tmdb_score])

In [12]:
df[['language', 'duration', 'tmdb_score']] = df['Title'].apply(lambda x: get_movie_details(str(x)))
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,language,duration,tmdb_score
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,en,103,6.254
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Drama Mystery,en,14,6.900
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,en,104,6.362
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,en,88,5.504
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,en,86,5.594
...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,en,103,6.300
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,en,90,4.305
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Drama Comedy,en,132,7.018
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,en,120,7.329


In [13]:
df.dtypes

Opening                object
Opening.1               int64
Title                  object
Production company     object
Cast and crew          object
Ref.                   object
genres                 object
language               object
duration                int64
tmdb_score            float64
dtype: object

In [14]:
df['duration'] = df['duration'].map('{:.1f}'.format)
df['tmdb_score'] = df['tmdb_score'].map('{:.1f}'.format)

In [15]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,language,duration,tmdb_score
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,en,103.0,6.3
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Drama Mystery,en,14.0,6.9
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,en,104.0,6.4
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,en,88.0,5.5
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,en,86.0,5.6
...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,en,103.0,6.3
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,en,90.0,4.3
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Drama Comedy,en,132.0,7.0
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,en,120.0,7.3


In [16]:
df['language'].value_counts().sort_index()

language
de      1
en    243
es      2
fr      1
he      1
hi      1
it      1
nl      1
Name: count, dtype: int64

In [17]:
language_mapping = {
    'de': 'German',
    'en': 'English',
    'es': 'Spanish',
    'fr': 'French',
    'he': 'Hebrew',
    'hi': 'Hindi',
    'it': 'Italian',
    'nl': 'Dutch',
}


df['language'] = df['language'].map(language_mapping)
df.rename(columns={'tmdb_score': 'imdb_score'}, inplace=True)

In [18]:
df

,Opening,Opening.1,Title,Production company,Cast and crew,Ref.,genres,language,duration,imdb_score
0,J A N U A R Y,5,Insidious: The Last Key,Universal Pictures / Blumhouse Productions / S...,Adam Robitel (director); Leigh Whannell (scree...,[2],Horror Thriller,English,103.0,6.3
1,J A N U A R Y,5,The Strange Ones,Vertical Entertainment,Christopher Radcliff (director/screenplay); La...,[3],Drama Mystery,English,14.0,6.9
2,J A N U A R Y,12,The Commuter,Lionsgate / StudioCanal / The Picture Company,Jaume Collet-Serra (director); Byron Willinger...,[4],Action Thriller Mystery,English,104.0,6.4
3,J A N U A R Y,12,Proud Mary,Screen Gems,"Babak Najafi (director); John S. Newman, Chris...",[5],Thriller Action Crime,English,88.0,5.5
4,J A N U A R Y,12,Acts of Violence,Lionsgate Premiere,Brett Donowho (director); Nicolas Aaron Mezzan...,[6],Action Crime Thriller,English,86.0,5.6
...,...,...,...,...,...,...,...,...,...,...
246,D E C E M B E R,21,Second Act,STX Entertainment,"Peter Segal (director); Justin Zackham, Elaine...",[237],Romance Comedy,English,103.0,6.3
247,D E C E M B E R,25,Holmes & Watson,Columbia Pictures / Gary Sanchez Productions /...,Etan Cohen (director/screenplay); Will Ferrell...,[141],Comedy Mystery Crime,English,90.0,4.3
248,D E C E M B E R,25,Vice,Annapurna Pictures / Plan B Entertainment,Adam McKay (director/screenplay); Christian Ba...,[116],Drama Comedy,English,132.0,7.0
249,D E C E M B E R,25,On the Basis of Sex,Focus Features,Mimi Leder (director); Daniel Stiepleman (scre...,[206],Drama History,English,120.0,7.3


In [19]:
df_2018 = df[['Title', 'Cast and crew', 'genres', 'language', 'duration', 'imdb_score']]
df_2018

,Title,Cast and crew,genres,language,duration,imdb_score
0,Insidious: The Last Key,Adam Robitel (director); Leigh Whannell (scree...,Horror Thriller,English,103.0,6.3
1,The Strange Ones,Christopher Radcliff (director/screenplay); La...,Drama Mystery,English,14.0,6.9
2,The Commuter,Jaume Collet-Serra (director); Byron Willinger...,Action Thriller Mystery,English,104.0,6.4
3,Proud Mary,"Babak Najafi (director); John S. Newman, Chris...",Thriller Action Crime,English,88.0,5.5
4,Acts of Violence,Brett Donowho (director); Nicolas Aaron Mezzan...,Action Crime Thriller,English,86.0,5.6
...,...,...,...,...,...,...
246,Second Act,"Peter Segal (director); Justin Zackham, Elaine...",Romance Comedy,English,103.0,6.3
247,Holmes & Watson,Etan Cohen (director/screenplay); Will Ferrell...,Comedy Mystery Crime,English,90.0,4.3
248,Vice,Adam McKay (director/screenplay); Christian Ba...,Drama Comedy,English,132.0,7.0
249,On the Basis of Sex,Mimi Leder (director); Daniel Stiepleman (scre...,Drama History,English,120.0,7.3


In [26]:
# Getting directors
def get_director(x):
    if " (director)" in x: #we are getting "director"
        return x.split(" (director)")[0]
    elif " (directors)" in x: #we are getting "directors" with "s"
        return x.split(" (directors)")[0]
    else:
        return x.split(" (director/screenplay)")[0] #we are getting "directors/screenplay"

df_2018['director_name'] = df_2018['Cast and crew'].map(lambda x: get_director(x))

In [27]:
# Getting Actor 1
def get_actor1(x):
    return ((x.split("screenplay); ")[-1]).split(", ")[0])

df_2018['actor_1_name'] = df_2018['Cast and crew'].map(lambda x: get_actor1(x))

In [28]:
# Getting Actor 2
def get_actor2(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 2:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[1])

df_2018['actor_2_name'] = df_2018['Cast and crew'].map(lambda x: get_actor2(x))


In [29]:
# Getting Actor 3
def get_actor3(x):
    if len((x.split("screenplay); ")[-1]).split(", ")) < 3:
        return np.nan
    else:
        return ((x.split("screenplay); ")[-1]).split(", ")[2])

df_2018['actor_3_name'] = df_2018['Cast and crew'].map(lambda x: get_actor3(x))

In [30]:
df_2018

,Title,Cast and crew,genres,language,duration,imdb_score,actor_1_name,actor_2_name,actor_3_name,director_name
0,Insidious: The Last Key,Adam Robitel (director); Leigh Whannell (scree...,Horror Thriller,English,103.0,6.3,Lin Shaye,Angus Sampson,Leigh Whannell,Adam Robitel
1,The Strange Ones,Christopher Radcliff (director/screenplay); La...,Drama Mystery,English,14.0,6.9,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Christopher Radcliff (director/screenplay); La...
2,The Commuter,Jaume Collet-Serra (director); Byron Willinger...,Action Thriller Mystery,English,104.0,6.4,Liam Neeson,Vera Farmiga,Patrick Wilson,Jaume Collet-Serra
3,Proud Mary,"Babak Najafi (director); John S. Newman, Chris...",Thriller Action Crime,English,88.0,5.5,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Babak Najafi
4,Acts of Violence,Brett Donowho (director); Nicolas Aaron Mezzan...,Action Crime Thriller,English,86.0,5.6,Bruce Willis,Cole Hauser,Shawn Ashmore,Brett Donowho
...,...,...,...,...,...,...,...,...,...,...
246,Second Act,"Peter Segal (director); Justin Zackham, Elaine...",Romance Comedy,English,103.0,6.3,Jennifer Lopez,Leah Remini,Vanessa Hudgens,Peter Segal
247,Holmes & Watson,Etan Cohen (director/screenplay); Will Ferrell...,Comedy Mystery Crime,English,90.0,4.3,Will Ferrell,John C. Reilly,Rebecca Hall,Etan Cohen
248,Vice,Adam McKay (director/screenplay); Christian Ba...,Drama Comedy,English,132.0,7.0,Christian Bale,Amy Adams,Steve Carell,Adam McKay
249,On the Basis of Sex,Mimi Leder (director); Daniel Stiepleman (scre...,Drama History,English,120.0,7.3,Felicity Jones,Armie Hammer,Justin Theroux,Mimi Leder


In [31]:
df_2018 = df_2018.rename(columns={'Title':'movie_title'})

In [32]:
new_df18 = df_2018.loc[:,['director_name','actor_1_name','actor_2_name','actor_3_name','genres','movie_title','language','duration','imdb_score']]

In [37]:
new_df18['actor_2_name'] = new_df18['actor_2_name'].replace(np.nan, 'unknown')
new_df18['actor_3_name'] = new_df18['actor_3_name'].replace(np.nan, 'unknown')

In [38]:
new_df18['movie_title'] = new_df18['movie_title'].str.lower()

In [39]:
new_df18['comb'] = new_df18['actor_1_name'] + ' ' + new_df18['actor_2_name'] + ' '+ new_df18['actor_3_name'] + ' '+ new_df18['director_name'] +' ' + new_df18['genres']
new_df18

,director_name,actor_1_name,actor_2_name,actor_3_name,genres,movie_title,language,duration,imdb_score,comb
0,Adam Robitel,Lin Shaye,Angus Sampson,Leigh Whannell,Horror Thriller,insidious: the last key,English,103.0,6.3,Lin Shaye Angus Sampson Leigh Whannell Adam Ro...
1,Christopher Radcliff (director/screenplay); La...,Lauren Wolkstein (director); Alex Pettyfer,James Freedson-Jackson,Emily Althaus,Drama Mystery,the strange ones,English,14.0,6.9,Lauren Wolkstein (director); Alex Pettyfer Jam...
2,Jaume Collet-Serra,Liam Neeson,Vera Farmiga,Patrick Wilson,Action Thriller Mystery,the commuter,English,104.0,6.4,Liam Neeson Vera Farmiga Patrick Wilson Jaume ...
3,Babak Najafi,Taraji P. Henson,Jahi Di'Allo Winston,Billy Brown,Thriller Action Crime,proud mary,English,88.0,5.5,Taraji P. Henson Jahi Di'Allo Winston Billy Br...
4,Brett Donowho,Bruce Willis,Cole Hauser,Shawn Ashmore,Action Crime Thriller,acts of violence,English,86.0,5.6,Bruce Willis Cole Hauser Shawn Ashmore Brett D...
...,...,...,...,...,...,...,...,...,...,...
246,Peter Segal,Jennifer Lopez,Leah Remini,Vanessa Hudgens,Romance Comedy,second act,English,103.0,6.3,Jennifer Lopez Leah Remini Vanessa Hudgens Pet...
247,Etan Cohen,Will Ferrell,John C. Reilly,Rebecca Hall,Comedy Mystery Crime,holmes & watson,English,90.0,4.3,Will Ferrell John C. Reilly Rebecca Hall Etan ...
248,Adam McKay,Christian Bale,Amy Adams,Steve Carell,Drama Comedy,vice,English,132.0,7.0,Christian Bale Amy Adams Steve Carell Adam McK...
249,Mimi Leder,Felicity Jones,Armie Hammer,Justin Theroux,Drama History,on the basis of sex,English,120.0,7.3,Felicity Jones Armie Hammer Justin Theroux Mim...


In [40]:
new_df18.to_csv('datasets/data3.csv')